## Reading json files

In [7]:
import json
from pathlib import Path

json_path = Path("../data/processed/parsed/cancer-clinical-studies.json")

if not json_path.exists():
    raise FileNotFoundError(f"Json file not found in {json_path}")

with open(json_path, "r", encoding="utf-8") as f:
    raw_json = json.load(f)
    
    
print(f"Top Level Keys : {list(raw_json.keys())}")
print(f"Total Document in JSON : {len(raw_json.get("documents", []))}")

# Get first document as a test 
sample_doc = raw_json["documents"][0]
print("\n---Sample Document Info---\n")
print("Keys : ", list(sample_doc.keys()))
print("File Name : ", sample_doc.get("file_name"))
print("Category : ", sample_doc.get("category"))
print("Total Pages : ", len(sample_doc.get("total_pages")))
print("PyMuPDF pages Count", (sample_doc.get("pymupdf_pages", [])))
print("Unstructured Elements count", len(sample_doc.get("unstructured_elements", [])))

# First Five elements of unstructured
print("\n---First 5 Unstructured Elements---\n")
for idx, el in enumerate(sample_doc.get("unstructured_elements", [])[:5]):
    print(f"[{idx}] Category : {el.get("category")}  | page = {el.get("metadata", {}).get("page_number")}")
    print(f"    text : {(el.get("text") or '')[:120].replace(chr(10), ' ')}...")

Top Level Keys : ['category_name', 'total_documents', 'successful_parses', 'failed_files', 'documents']
Total Document in JSON : 52

---Sample Document Info---

Keys :  ['file_name', 'metadata', 'total_pages', 'unstructured_elements', 'category']
File Name :  A c-MET-Targeted Topical Fluorescent Probe cMBP-ICG Improves Oral Squamous Cell Carcinoma Detection in Humans.pdf
Category :  cancer-clinical-studies
Total Pages :  11
PyMuPDF pages Count []
Unstructured Elements count 532

---First 5 Unstructured Elements---

[0] Category : Header  | page = 1
    text : Ann Surg Oncol (2023) 30:641–651 https://doi.org/10.1245/s10434-022-12532-x...
[1] Category : Image  | page = 1
    text : Annals of SURGICALONCOLOGY OFFICIAL JOURNAL OF THE SOCIETY OF SURGICAL ONCOLOGY...
[2] Category : Image  | page = 1
    text : ® “dae updates...
[3] Category : NarrativeText  | page = 1
    text : ORIGINAL ARTICLE – TRANSLATIONAL RESEARCH...
[4] Category : Title  | page = 1
    text : A c-MET-Targeted Topical 

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

pages_list = sample_doc.get("total_pages", [])
full_text = "\n\n".join(
    [f"[Page {p.get("page_number")}]\n{p.get("text", '')}" for p in pages_list]
)

# Recursive Settings
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n", "\n\n", ".", " ", ""],
    length_function=len
    )

recursive_chunks = recursive_splitter.split_text(full_text)

print(f"Total Characters in document : {len(full_text)}")
print(f"Number of Recursive Chunks : {len(recursive_chunks)}")

lengths = [len(c) for c in recursive_chunks]

print(f"Chunk Lengths  -> min: {min(lengths)}, avg: {sum(lengths)//len(lengths)}, max: {max(lengths)}")

print("\n---Chunk 0---\n")
print(recursive_chunks[0])

print("\n---Chunk 4---\n")
print(recursive_chunks[4])

Total Characters in document : 39791
Number of Recursive Chunks : 59
Chunk Lengths  -> min: 462, avg: 769, max: 799

---Chunk 0---

[Page 1]
ORIGINAL ARTICLE – TRANSLATIONAL RESEARCH
A c-MET-Targeted Topical Fluorescent Probe cMBP-ICG
Improves Oral Squamous Cell Carcinoma Detection in Humans
Jingbo Wang, MD1
, Siyi Li, MD2, Kun Wang, PhD3, Ling Zhu, MD1, Lin Yang, MD1,
Yunjing Zhu, MD1, Zhen Zhang, MD2, Longwei Hu, MD2, Yuan Yuan, MD1, Qi Fan, MD1,
Jiliang Ren, MD1, Gongxin Yang, MD1, Weilong Ding, BS1, Xiaoyu Zhou, BS4, Junqi Cui, BS5,
Chunye Zhang, MD5, Ying Yuan, MD1, Ruimin Huang, PhD4, Jie Tian, PhD3, and
Xiaofeng Tao, MD1
1Department of Radiology, Ninth People’s Hospital, School of Medicine, Shanghai Jiao Tong University, Shanghai, China;
2Department of Oral and Maxillofacial-Head and Neck Oncology, Ninth People’s Hospital, School of Medicine, Shanghai

---Chunk 4---

and improve oncologic outcomes.
Approximately 95% of all head and neck cancers are
squamous cell carcinoma (SCC),

In [13]:
IGNORED_CATEGORIES = {"Header", "Footer", "PageNumber", "Image"}
MAX_CHUNK_CHARS = 850

element_chunks = []
current_parts = []
current_len = 0
current_meta = {"pages": set(), "categories": set()}

def flush_chunk():
    global current_parts, current_meta, current_len
    if not current_parts:
        return

    element_chunks.append({
        "text": "\n\n".join(current_parts).strip(),
        "pages": sorted(p for p in current_meta["pages"] if p is not None),
        "categories": sorted(current_meta["categories"]),
    })
    current_parts = []
    current_len = 0
    current_meta = {"pages": set(), "categories": set()}

for el in sample_doc.get("unstructured_elements", []):
    cat = el.get("category", "Uncategorized")
    text = (el.get("text") or "").strip()
    page = el.get("metadata", {}).get("page_number")

    if not text or cat in IGNORED_CATEGORIES:
        continue

    if cat in {"Table", "FigureCaption"}:
        flush_chunk()
        html = el.get("metadata", {}).get("text_as_html")
        body = f"HTML Table:\n{html}\n\nPlain Text:\n{text}" if (cat == "Table" and html) else text
        element_chunks.append({
            "text": f"[{cat} | Page {page}]\n{body}".strip(),
            "pages": [page] if page is not None else [],
            "categories": [cat],
        })
        continue

    added_len = len(text) + (2 if current_parts else 0)
    if current_parts and (current_len + added_len > MAX_CHUNK_CHARS):
        flush_chunk()

    current_parts.append(text)
    current_len += added_len
    if page is not None:
        current_meta["pages"].add(page)
    current_meta["categories"].add(cat)

flush_chunk()

print(f"Number of elements chunks : {len(element_chunks)}")
el_lengths = [len(c["text"]) for c in element_chunks]
print(f"Chunk lengths -> min: {min(el_lengths)}, avg: {sum(el_lengths)//len(el_lengths)}, max: {max(el_lengths)}")

print("\n--- Element Chunk [0] ---")
print(element_chunks[0])

print("\n--- Element Chunk [4] ---")
print(element_chunks[4])

Number of elements chunks : 67
Chunk lengths -> min: 8, avg: 596, max: 1099

--- Element Chunk [0] ---
{'text': 'ORIGINAL ARTICLE – TRANSLATIONAL RESEARCH\n\nA c-MET-Targeted Topical Fluorescent Probe cMBP-ICG Improves Oral Squamous Cell Carcinoma Detection in Humans\n\nJingbo Wang, MD1 , Siyi Li, MD2, Kun Wang, PhD3, Ling Zhu, MD1, Lin Yang, MD1, Yunjing Zhu, MD1, Zhen Zhang, MD2, Longwei Hu, MD2, Yuan Yuan, MD1, Qi Fan, MD1, Jiliang Ren, MD1, Gongxin Yang, MD1, Weilong Ding, BS1, Xiaoyu Zhou, BS4, Junqi Cui, BS5, Chunye Zhang, MD5, Ying Yuan, MD1, Ruimin Huang, PhD4, Jie Tian, PhD3, and Xiaofeng Tao, MD1', 'pages': [1], 'categories': ['NarrativeText', 'Title']}

--- Element Chunk [4] ---
{'text': 'Y. Yuan, MD e-mail: yuany83@163.com\n\nR. Huang, PhD e-mail: rmhuang@simm.ac.cn\n\nJ. Tian, PhD e-mail: tian@ieee.org\n\nApproximately 95% of all head and neck cancers are squamous cell carcinoma (SCC), the most common of which is oral SCC (excluding non-melanoma cutaneous cancers).1 Oral S

In [14]:
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter

IGNORED_CATEGORIES = {"Header", "Footer", "PageNumber", "Image"}
SECTION_PATTERN = re.compile(
    r"^(abstract|introduction|background|methods|materials and methods|results|discussion|conclusions?|case report|references)\b",
    re.IGNORECASE,
)

sub_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
)

section_chunks = []
current_section = "General / Header"
section_buffer = []
section_pages = set()

def flush_section():
    global section_buffer, section_pages
    raw_sec_text = "\n\n".join(section_buffer).strip()
    if not raw_sec_text:
        section_buffer = []
        section_pages = set()
        return

    parts = sub_splitter.split_text(raw_sec_text)
    pages_list = sorted(p for p in section_pages if p is not None)
    for i, part in enumerate(parts):
        if len(part.strip()) < 25 and len(parts) > 1:
            continue
        section_chunks.append({
            "section": current_section,
            "part_index": i,
            "pages": pages_list,
            "text": part.strip(),
        })

    section_buffer = []
    section_pages = set()

for el in sample_doc.get("unstructured_elements", []):
    cat = el.get("category", "Uncategorized")
    text = (el.get("text") or "").strip()
    page = el.get("metadata", {}).get("page_number")

    if not text or cat in IGNORED_CATEGORIES:
        continue

    first_line = text.splitlines()[0].strip()
    is_heading = (cat == "Title") or (len(first_line) < 70 and SECTION_PATTERN.match(first_line))

    if is_heading and len(text) < 120:
        flush_section()
        current_section = first_line
        if page is not None:
            section_pages.add(page)
        continue

    section_buffer.append(text)
    if page is not None:
        section_pages.add(page)

flush_section()

print(f"Number of section chunks: {len(section_chunks)}")
sec_lengths = [len(c["text"]) for c in section_chunks]
print(f"Chunk lengths -> min: {min(sec_lengths)}, avg: {sum(sec_lengths)//len(sec_lengths)}, max: {max(sec_lengths)}")

print("\nDetected sections:", sorted({c["section"] for c in section_chunks}))

for c in section_chunks[:4]:
    print(f"\n--- Section: {c['section']} | Pages: {c['pages']} | Len: {len(c['text'])} ---")
    print(c["text"][:350])

Number of section chunks: 67
Chunk lengths -> min: 41, avg: 594, max: 798

Detected sections: ['A c-MET-Targeted Topical Fluorescent Probe cMBP-ICG Improves Oral Squamous Cell Carcinoma Detection in Humans', 'ABSTRACT', 'CONCLUSION', 'DISCUSSION', 'General / Header', 'METHODS', 'REFERENCES', 'RESULTS']

--- Section: General / Header | Pages: [1] | Len: 41 ---
ORIGINAL ARTICLE – TRANSLATIONAL RESEARCH

--- Section: A c-MET-Targeted Topical Fluorescent Probe cMBP-ICG Improves Oral Squamous Cell Carcinoma Detection in Humans | Pages: [1] | Len: 342 ---
Jingbo Wang, MD1 , Siyi Li, MD2, Kun Wang, PhD3, Ling Zhu, MD1, Lin Yang, MD1, Yunjing Zhu, MD1, Zhen Zhang, MD2, Longwei Hu, MD2, Yuan Yuan, MD1, Qi Fan, MD1, Jiliang Ren, MD1, Gongxin Yang, MD1, Weilong Ding, BS1, Xiaoyu Zhou, BS4, Junqi Cui, BS5, Chunye Zhang, MD5, Ying Yuan, MD1, Ruimin Huang, PhD4, Jie Tian, PhD3, and Xiaofeng Tao, MD1

--- Section: A c-MET-Targeted Topical Fluorescent Probe cMBP-ICG Improves Oral Squamous Cell Carcino

In [15]:
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter

IGNORED_CATEGORIES = {"Header", "Footer", "PageNumber", "Image"}
SECTION_PATTERN = re.compile(
    r"^(abstract|introduction|background|methods|materials and methods|results|discussion|conclusions?|case report|references)\b",
    re.IGNORECASE,
)

sub_splitter = RecursiveCharacterTextSplitter(
    chunk_size=750,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
)

doc_title = sample_doc.get("file_name", "Unknown").replace(".pdf", "")
doc_category = sample_doc.get("category", "Unknown")

enriched_chunks = []
current_section = "Front Matter"
section_buffer = []
section_pages = set()

def flush_enriched():
    global section_buffer, section_pages
    raw_sec_text = "\n\n".join(section_buffer).strip()
    section_buffer = []
    pages_list = sorted(p for p in section_pages if p is not None)
    section_pages = set()

    # استبعاد قسم المراجع لتقليل الضوضاء في الاسترجاع
    if not raw_sec_text or current_section.strip().lower().startswith("reference"):
        return

    parts = sub_splitter.split_text(raw_sec_text)
    for part in parts:
        clean_part = part.strip()
        if len(clean_part) < 120:
            continue

        header = (
            f"[Doc: {doc_title} | Category: {doc_category} | "
            f"Section: {current_section} | Pages: {pages_list}]"
        )
        enriched_chunks.append({
            "text": f"{header}\n{clean_part}",
            "metadata": {
                "file_name": sample_doc.get("file_name"),
                "category": doc_category,
                "section": current_section,
                "pages": pages_list,
            },
        })

for el in sample_doc.get("unstructured_elements", []):
    cat = el.get("category", "Uncategorized")
    text = (el.get("text") or "").strip()
    page = el.get("metadata", {}).get("page_number")

    if not text or cat in IGNORED_CATEGORIES:
        continue

    first_line = text.splitlines()[0].strip()
    is_heading = (cat == "Title") or (len(first_line) < 70 and SECTION_PATTERN.match(first_line))

    if is_heading and len(text) < 120:
        flush_enriched()
        current_section = first_line
        if page is not None:
            section_pages.add(page)
        continue

    section_buffer.append(text)
    if page is not None:
        section_pages.add(page)

flush_enriched()

print(f"Number of enriched chunks: {len(enriched_chunks)}")
en_lengths = [len(c["text"]) for c in enriched_chunks]
print(f"Chunk lengths -> min: {min(en_lengths)}, avg: {sum(en_lengths)//len(en_lengths)}, max: {max(en_lengths)}")

print("\n--- Enriched Chunk [0] ---")
print(enriched_chunks[0]["text"][:500])

print("\n--- Enriched Chunk [3] ---")
print(enriched_chunks[3]["text"][:500])

Number of enriched chunks: 61
Chunk lengths -> min: 339, avg: 744, max: 1018

--- Enriched Chunk [0] ---
[Doc: A c-MET-Targeted Topical Fluorescent Probe cMBP-ICG Improves Oral Squamous Cell Carcinoma Detection in Humans | Category: cancer-clinical-studies | Section: A c-MET-Targeted Topical Fluorescent Probe cMBP-ICG Improves Oral Squamous Cell Carcinoma Detection in Humans | Pages: [1]]
Jingbo Wang, MD1 , Siyi Li, MD2, Kun Wang, PhD3, Ling Zhu, MD1, Lin Yang, MD1, Yunjing Zhu, MD1, Zhen Zhang, MD2, Longwei Hu, MD2, Yuan Yuan, MD1, Qi Fan, MD1, Jiliang Ren, MD1, Gongxin Yang, MD1, Weilong Ding, BS

--- Enriched Chunk [3] ---
[Doc: A c-MET-Targeted Topical Fluorescent Probe cMBP-ICG Improves Oral Squamous Cell Carcinoma Detection in Humans | Category: cancer-clinical-studies | Section: ABSTRACT | Pages: [1, 2]]
conducted with oral SCC patients and intraoperatively to assess the feasibility of cMBP-ICG used to detect tumors margins. Fluorescence was histologically correlated to determin

In [16]:
import re

IGNORED_CATEGORIES = {"Header", "Footer", "PageNumber", "Image"}

clean_paragraphs = []
for el in sample_doc.get("unstructured_elements", []):
    cat = el.get("category", "Uncategorized")
    text = (el.get("text") or "").strip()
    if not text or cat in IGNORED_CATEGORIES:
        continue
    if len(re.findall(r"\b(?:MD|PhD|BS|MS)\b", text)) >= 3:
        continue
    clean_paragraphs.append(text)

raw_text = "\n\n".join(clean_paragraphs)
sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", raw_text) if len(s.strip()) > 10]

MAX_CHARS = 800
OVERLAP_SENTENCES = 2

sentence_window_chunks = []
current_sents = []

for sent in sentences:
    current_len = sum(len(s) for s in current_sents) + max(0, len(current_sents) - 1)
    if current_sents and (current_len + len(sent) + 1 > MAX_CHARS):
        sentence_window_chunks.append(" ".join(current_sents).strip())
        current_sents = current_sents[-OVERLAP_SENTENCES:] if OVERLAP_SENTENCES > 0 else []
    current_sents.append(sent)

if current_sents:
    sentence_window_chunks.append(" ".join(current_sents).strip())

print(f"Total sentences: {len(sentences)}")
print(f"Number of sentence-window chunks: {len(sentence_window_chunks)}")
sw_lengths = [len(c) for c in sentence_window_chunks]
print(f"Chunk lengths -> min: {min(sw_lengths)}, avg: {sum(sw_lengths)//len(sw_lengths)}, max: {max(sw_lengths)}")

print("\n--- Sentence-Window Chunk [0] ---")
print(sentence_window_chunks[0][:500])

print("\n--- Sentence-Window Chunk [2] ---")
print(sentence_window_chunks[2][:500])

Total sentences: 326
Number of sentence-window chunks: 87
Chunk lengths -> min: 401, avg: 816, max: 1914

--- Sentence-Window Chunk [0] ---
ORIGINAL ARTICLE – TRANSLATIONAL RESEARCH

A c-MET-Targeted Topical Fluorescent Probe cMBP-ICG Improves Oral Squamous Cell Carcinoma Detection in Humans

Jingbo Wang, MD1 , Siyi Li, MD2, Kun Wang, PhD3, Ling Zhu, MD1, Lin Yang, MD1, Yunjing Zhu, MD1, Zhen Zhang, MD2, Longwei Hu, MD2, Yuan Yuan, MD1, Qi Fan, MD1, Jiliang Ren, MD1, Gongxin Yang, MD1, Weilong Ding, BS1, Xiaoyu Zhou, BS4, Junqi Cui, BS5, Chunye Zhang, MD5, Ying Yuan, MD1, Ruimin Huang, PhD4, Jie Tian, PhD3, and Xiaofeng Tao, MD1

1D

--- Sentence-Window Chunk [2] ---
ORIGINAL ARTICLE – TRANSLATIONAL RESEARCH

A c-MET-Targeted Topical Fluorescent Probe cMBP-ICG Improves Oral Squamous Cell Carcinoma Detection in Humans

Jingbo Wang, MD1 , Siyi Li, MD2, Kun Wang, PhD3, Ling Zhu, MD1, Lin Yang, MD1, Yunjing Zhu, MD1, Zhen Zhang, MD2, Longwei Hu, MD2, Yuan Yuan, MD1, Qi Fan, MD1, Jiliang Re

In [17]:
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter

IGNORED_CATEGORIES = {"Header", "Footer", "PageNumber", "Image"}

# تجميع عناصر نصية نظيفة مع استبعاد سطور المؤلفين الواضحة
clean_blocks = []
for el in sample_doc.get("unstructured_elements", []):
    cat = el.get("category", "Uncategorized")
    text = (el.get("text") or "").strip()
    page = el.get("metadata", {}).get("page_number")
    if not text or cat in IGNORED_CATEGORIES:
        continue
    if len(re.findall(r"\b(?:MD|PhD|BS|MS)\b", text)) >= 3:
        continue
    clean_blocks.append({"text": text, "page": page, "category": cat})

full_text = "\n\n".join(b["text"] for b in clean_blocks)

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=60,
    separators=["\n\n", "\n", ". ", " ", ""],
)

parent_docs = parent_splitter.split_text(full_text)
parent_chunks = []
child_chunks = []

for p_idx, p_text in enumerate(parent_docs):
    p_clean = p_text.strip()
    if len(p_clean) < 100:
        continue

    parent_id = f"parent_{p_idx}"
    parent_chunks.append({
        "parent_id": parent_id,
        "text": p_clean,
    })

    sub_texts = child_splitter.split_text(p_clean)
    for c_idx, c_text in enumerate(sub_texts):
        c_clean = c_text.strip()
        if len(c_clean) < 40:
            continue
        child_chunks.append({
            "child_id": f"{parent_id}_child_{c_idx}",
            "parent_id": parent_id,
            "text": c_clean,
        })

p_lengths = [len(p["text"]) for p in parent_chunks]
c_lengths = [len(c["text"]) for c in child_chunks]

print(f"Parent chunks count: {len(parent_chunks)}")
print(f"Parent lengths -> min: {min(p_lengths)}, avg: {sum(p_lengths)//len(p_lengths)}, max: {max(p_lengths)}")

print(f"\nChild chunks count:  {len(child_chunks)}")
print(f"Child lengths  -> min: {min(c_lengths)}, avg: {sum(c_lengths)//len(c_lengths)}, max: {max(c_lengths)}")

print("\n--- Sample Parent [1] ---")
print(parent_chunks[1]["text"][:400])

print("\n--- Children of Parent [1] ---")
linked_children = [c for c in child_chunks if c["parent_id"] == parent_chunks[1]["parent_id"]]
for ch in linked_children[:3]:
    print(f"[{ch['child_id']}]: {ch['text']}\n")

Parent chunks count: 42
Parent lengths -> min: 496, avg: 966, max: 1196

Child chunks count:  182
Child lengths  -> min: 46, avg: 227, max: 350

--- Sample Parent [1] ---
1Department of Radiology, Ninth People’s Hospital, School of Medicine, Shanghai Jiao Tong University, Shanghai, China; 2Department of Oral and Maxillofacial-Head and Neck Oncology, Ninth People’s Hospital, School of Medicine, Shanghai Jiao Tong University, Shanghai, China; 3CAS Key Laboratory of Molecular Imaging, Beijing Key Laboratory of Molecular Imaging, The State Key Laboratory of Management 

--- Children of Parent [1] ---
[parent_1_child_0]: 1Department of Radiology, Ninth People’s Hospital, School of Medicine, Shanghai Jiao Tong University, Shanghai, China; 2Department of Oral and Maxillofacial-Head and Neck Oncology, Ninth People’s Hospital, School of Medicine, Shanghai Jiao Tong University, Shanghai, China; 3CAS Key Laboratory of Molecular Imaging, Beijing Key Laboratory of

[parent_1_child_1]: Laboratory of

# Chunking Strategy — Evaluation & Final Decision

## 1. Objective
Evaluate and compare different text chunking strategies on parsed scientific oncology PDFs (structured JSON outputs from the hybrid `PyMuPDF + Unstructured [hi_res]` pipeline) to select the optimal approach for a text-only medical RAG pipeline.

---

## 2. Evaluated Chunking Strategies

Seven chunking strategies were implemented and tested on a representative clinical study (`A c-MET-Targeted Topical Fluorescent Probe cMBP-ICG Improves Oral Squamous Cell Carcinoma Detection in Humans.pdf`):

1. **Recursive Character Chunking** (`chunk_size=800`, `overlap=120`)
2. **Element-Aware Chunking** (based on `Unstructured` element categories)
3. **Section / Heading-Aware Chunking** (grouping by scientific paper sections)
4. **Context-Enriched Chunking** (prepending document/section metadata headers)
5. **Semantic Chunking** (embedding cosine-similarity breakpoints)
6. **Sentence-Window Chunking** (sentence boundary splitting with sentence overlap)
7. **Parent-Document (Small-to-Big) Chunking** (1200-char parent / 350-char child)

---

## 3. Evaluation Results

| Strategy | Chunk Count | Length (Min / Avg / Max) | Key Observations & Limitations |
| :--- | :---: | :---: | :--- |
| **Recursive Character** | 59 | 462 / 769 / 799 | Fast and uniform, but cuts mid-sentence and mixes multi-column noise (authors, affiliations, emails, DOI) with body text. |
| **Element-Aware** | 67 | 8 / 596 / 1099 | Successfully isolates `Table` and `FigureCaption` elements, but produces tiny fragments (8 chars), exceeds max size, and retains sidebar metadata. |
| **Section-Aware** | 67 | 41 / 594 / 798 | Accurately detects major sections (`ABSTRACT`, `METHODS`, `RESULTS`, `DISCUSSION`, `CONCLUSION`), but retains author blocks, `REFERENCES`, and tiny header fragments. |
| **Context-Enriched** | 61 | 339 / 744 / 1018 | Balanced chunk sizes and clear contextual headers (`Doc`, `Category`, `Section`, `Pages`), eliminating tiny fragments; still requires filtering author/affiliation blocks. |
| **Semantic** | Variable | Sentence-based | Captures topic shifts via embeddings, but slower to compute and risks splitting closely linked clinical facts (e.g., dosage vs. outcome). |
| **Sentence-Window** | 87 | 401 / 816 / 1914 | Preserves sentence boundaries, but repeated long author/affiliation blocks via overlap and spiked max length to 1914 chars. |
| **Parent-Document** | 42 P / 182 C | Child: 46 / 227 / 350 | High indexing overhead (~57k child chunks across 314 PDFs) and still polluted by affiliation blocks without prior cleaning. |

---

## 4. Final Architecture Decision

No single standalone strategy handled multi-column scientific papers cleanly without noise. We selected a **Hybrid Chunking Pipeline** combining the strengths of the top-performing approaches:

1. **Noise Filtering (Pre-Chunking Cleaning):**
   - Exclude non-body elements (`Header`, `Footer`, `PageNumber`, `Image`).
   - Filter out author/affiliation blocks and `REFERENCES` sections to prevent retrieval pollution.
2. **Section-Aware & Element-Aware Splitting:**
   - Group narrative content under detected section headings (`ABSTRACT`, `INTRODUCTION`, `METHODS`, `RESULTS`, `DISCUSSION`, `CONCLUSION`).
   - Keep `Table` (with HTML + plain text) and `FigureCaption` elements intact as standalone chunks to preserve clinical data integrity.
   - Apply recursive sub-splitting (`~750–850` chars, `100` overlap) only when a section exceeds the target limit, enforcing a minimum chunk threshold (`>= 120` chars).
3. **Context Enrichment:**
   - Prepend a standardized metadata header to every chunk:
     `[Doc: <title> | Category: <category> | Section: <section> | Pages: <pages>]`

---

## 5. Future Roadmap (Project Developing Phase)
- **Multimodal UI / Visual Retrieval:** Retain extracted figures and link them via `FigureCaption` metadata so the UI can display the relevant chart/scan alongside the text-only LLM response.

---

## 6. Next Step
Implement the final **Hybrid Chunking Pipeline** script, validate its output on the sample document, and run batch chunking across all category JSON files prior to embedding and vector storage.